# Modellering af ikke-lineære detailefterspørgselskurver med PROC GAM

## Resumé

Denne notebook bruger **PROC GAM** til at modellere ugentligt dagligvaresalg som en glat, ikke-lineær funktion af hyldepris, butikstemperatur (en sæsonproxy) og kampagneudgifter, med en parametrisk effekt af kampagneflaget. Generaliserede additive modeller lader en kategorichef afdække de sande kurvede prisfølsomheds- og sæsonefterspørgselsformer, som en lineær regression ville overse, hvilket understøtter skarpere pris- og kampagnebeslutninger.

## Datakilder

| Datasæt | Rækker | Granularitet | Nøglevariable | Beskrivelse |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | uge | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Syntetisk ugentlig salgsserie for én flagskibsdagligvarebutik over 100 fortløbende uger (næsten to sæsoncyklusser). Genereret direkte med `call streaminit(20250531)` og `rand()`. Ugentligt enhedssalg følger en Poisson-efterspørgselsproces, hvor middelværdien drives af en eksponentiel priskurve, en kvadratisk temperatur-/sæsoneffekt med top nær 72F, og et konkavt (kvadratrods-)løft fra kampagneudgifter, plus et diskret kampagneugeflag. |

# Modellering af ikke-lineære detailefterspørgselskurver med PROC GAM

Detailefterspørgsel reagerer sjældent lineært på pris, vejr eller kampagneudgifter. En lille prisnedsættelse på en basisvare flytter måske knap nok volumen, mens en overskridelse af et psykologisk prispunkt kan udløse et pludseligt hop; vejrdrevet efterspørgsel topper i et behageligt mellemleje og falder i begge yderpunkter; og kampagneløft viser aftagende afkast, efterhånden som udgifterne stiger.

**PROC GAM** (generaliserede additive modeller) tilpasser hver driver med sit eget glatte spline-led, så det er dataene — ikke en rigid lineær antagelse — der bestemmer formen på hver efterspørgselskurve. Her modellerer vi ugentligt enhedssalg for én flagskibsdagligvarebutik over 100 fortløbende uger, ved at kombinere et parametrisk kampagneflag med udglatningssplines på pris, temperatur og kampagneudgifter under en Poisson-respons.

## Trin 1 — Generér en syntetisk ugentlig salgsserie

Vi simulerer 100 fortløbende uger (næsten to sæsoncyklusser) af salgsdata for én flagskibsbutik. Den dataskabende proces er bevidst ikke-lineær, så vi kan bekræfte, at GAM genfinder realistiske former:

- **Price** driver volumen gennem en eksponentiel responskurve (`exp(-1.1 * Price)`), så efterspørgslen stiger stejlt, når prisen falder.
- **Temperature** fungerer som en sæsonproxy med en kvadratisk top nær 72F, som modellerer komfortdrevet kundetrafik.
- **PromoSpend** giver et konkavt kvadratrodsløft (aftagende afkast).
- Et diskret **Promotion**-flag tilføjer en parametrisk trineffekt i kampagneuger.

Ugentlige `Units` trækkes fra en Poisson-fordeling, hvilket matcher optællingskarakteren af enhedssalg.

In [1]:
data store_sales;
   CALL streaminit(20250531);
   LÆNGDE Promotion $3;
   GØR Week = 1 TIL 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      HVIS rand("uniform") < 0.28 SÅ GØR;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      SLUT;
      ELLERS GØR;
         Promotion  = "No";
         PromoSpend = 0;
      SLUT;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      UDDATA;
   SLUT;
KØR;


NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.03 seconds
  cpu   0.03 seconds


## Trin 2 — Profilér de simulerede data

En hurtig `PROC MEANS` bekræfter, at variablene ligger i fornuftige detailintervaller før modellering: enhedstal er ikke-negative heltal, prisen samler sig omkring nogle få dollars, temperaturen dækker en hel sæsoncyklus, og kampagneudgifterne er nul i uger uden kampagne.

In [2]:
PROCEDURE GENNEMSNIT data=store_sales n mean MIN MAX maxdec=2;
   VARIABEL Units Price Temperature PromoSpend;
   MÆRKAT Units="Enheder" Price="Pris" Temperature="Temperatur" PromoSpend="Kampagneudgift";
KØR;

                                                  The MEANS Procedure

 Variable     Label                  N           Mean     Minimum     Maximum
 ----------------------------------------------------------------------------
 Units        Enheder              100           7.03        0.00      103.00
 Price        Pris                 100           3.15        2.81        3.48
 Temperature  Temperatur           100          55.50       22.72       83.49
 PromoSpend   Kampagneudgift       100         128.76        0.00      779.00
 ----------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Trin 3 — Tilpas den fulde additive efterspørgselsmodel

Kernemodellen kombinerer:

- `param(Promotion)` — en parametrisk (lineær) effekt for kampagneugeindikatoren, deklareret i `CLASS`-sætningen.
- `spline(Price, df=4)` — en kubisk udglatningsspline, der fanger den kurvede prisrespons.
- `spline(Temperature, df=4)` — en glat sæsonkurve.
- `spline(PromoSpend, df=3)` — kampagneløft med aftagende afkast.

Fordi enhedssalg er optællinger, angiver vi `dist=poisson` (log-link). `method=gcv` lader generaliseret krydsvalidering styre udglatningsgraden af hver komponent uden en eksplicit tilsidesættelse af frihedsgrader. `OUTPUT`-sætningen skriver forudsigelser og residualer pr. observation til `gam_fit`.

Proceduren rapporterer en **Deviance på 43.59** mod en **Null Deviance på 2041.12** — de fire glatte plus parametriske drivere forklarer næsten al den variation, en model med kun et konstantled ville lade tilbage — og en **AIC på 254.61**. Det parametriske `PROMOTIONYES`-estimat er positivt (+0.41 på logskalaen), hvilket bekræfter kampagnevolumenløftet, og prisens spline har en stærkt negativ lineær tendens (−1.71), signaturen på faldende efterspørgsel.

In [3]:
PROCEDURE gam data=store_sales PLOTS=components(CLM commonaxes);
   KLASSE Promotion;
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   MÆRKAT Units="Enheder" Price="Pris" Temperature="Temperatur" PromoSpend="Kampagneudgift" Promotion="Kampagne";
   UDDATA out=gam_fit predicted residual;
KØR;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Enheder
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Pris)                     4.0000         


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Trin 4 — En fokuseret pris- og sæsonmodel

For et slankere prisgennemsyn genfitter vi med kun de to mest beslutningsrelevante glatte drivere — pris og temperatur — og giver prisen ekstra fleksibilitet (`df=5`) til at opløse eventuelle knæk nær et psykologisk prispunkt. Kampagneflaget bevares som en parametrisk kontrol.

Når kampagneudgifter droppes, stiger **Deviance til 62.80** og **AIC til 269.82**, begge højere end den fulde models 43.59 og 254.61. Det parametriske `PROMOTIONYES`-led absorberer også mere af kampagnesignalet her (+1.73 mod +0.41), da udgiftssplinen ikke længere er til stede til at bære det. Prissplinen bevarer sin negative tendens (−1.51), så kernefortællingen om efterspørgslen er stabil på tværs af specifikationer.

In [4]:
PROCEDURE gam data=store_sales PLOTS=components(CLM);
   KLASSE Promotion;
   MODEL Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   MÆRKAT Units="Enheder" Price="Pris" Temperature="Temperatur" Promotion="Kampagne";
KØR;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Enheder
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Pris)                     5.0000         5.0000
Spline(Temperatur)               4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Fortolkning af resultaterne

Tabellen **Regression Model Analysis** rapporterer den lineære tendens inden for hver komponent plus den parametriske kampagneeffekt. Den positive `PROMOTIONYES`-koefficient (+0.41 i den fulde model, +1.73 i den slankere) bekræfter det forventede kampagnevolumenløft, mens den negative lineære tendens på prissplinen (−1.71 og −1.51) afspejler klassisk faldende efterspørgsel. Temperatursplinens lille positive lineære led (+0.03) er forventet: dens egentlige historie er krumningen omkring toppen ved 72F, som en enkelt lineær koefficient ikke kan opsummere.

Tabellen **Smoothing Model Analysis** rapporterer de frihedsgrader, hver spline forbruger. Pris og temperatur trækker hver 4 effektive frihedsgrader (5 for pris i den slankere model) og kampagneudgifter 3 — hver langt over den ene frihedsgrad, en ret linje ville bruge, hvilket er præcis grunden til, at en lineær regression ville overse disse kurvede sammenhænge.

**Fit Statistics** lader dig sammenligne de to specifikationer direkte. Den fulde firedriver-model opnår en Deviance på 43.59 og AIC på 254.61 mod den slankere models 62.80 og 269.82; begge kriterier favoriserer den fulde model, hvilket viser, at kampagneudgifter og kampagneflaget tilføjer forklaringskraft ud over pris og temperatur alene. I forhold til Null Deviance på 2041.12 fanger begge modeller langt størstedelen af efterspørgselsvariationen.

Sammen giver disse tabeller en kategorichef en kvantificeret, datadrevet efterspørgselsfortælling: en stejl prisrespons, der informerer nedskrivningsdybden, et sæsonbestemt temperaturvindue og en kampagneudgiftseffekt med aftagende afkast — langt skarpere vejledning end et enkelt lineært elasticitetsestimat. (PROC GAM accepterer også `plots=components` til at tegne de partielle prognosekurver for hvert glat led; de numeriske komponenttabeller ovenfor er kilden, disse kurver er tegnet ud fra.)